In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings , ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [57]:
loader = PyPDFLoader("../Data/0121YG008021121_190858c.pdf")
docs = loader.load()
len(docs)

11

In [72]:
splitter = RecursiveCharacterTextSplitter(chunk_size=10000,chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

11

In [59]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [60]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [61]:
query = "Machine Learning and Data Science Content"
data = vector_store.similarity_search(query=query)

In [9]:
len(data)

4

In [10]:
data[0]

Document(id='cd7c935f-208f-44a9-9c8d-2e32cf523928', metadata={'producer': 'iTextSharp™ 5.5.13.3 ©2000-2022 iText Group NV (AGPL-version)', 'page': 1, 'moddate': '2025-07-24T17:55:07+05:30', 'source': '../Data/0121YG008021121_190858c.pdf', 'creationdate': '2025-07-24T17:55:07+05:30', 'page_label': '2', 'creator': 'PyPDF', 'total_pages': 11}, page_content='(Reference to -  The diagnostic and predictive role of NLR, d-NLR and PLR in COVID-19 patients   A.-P. Yang, et al.  International Immunopharmacology 84 (2020) 106504\nThis ratio element is a calculated parameter and out of NABL scope.\nView Details View Report\nPage 2 Of 11\nDr. Shavi Mahajan\nLabhead-Pathologist\nULR No.775000013478100-0121\nAgilus Diagnostics Ltd\nAstro Healthcare R&D Pvt. Ltd., Lab 28-C, 2nd Extension, Opp. Power Grid Corporation,\nGandhi Nagar,Jammu Tawi\nJammu, 180004\nJammu & Kashmir, India\nTel : 0191-2479435,0191-2479439\n918899054002, Fax : \nCIN - U74899PB1995PLC045956\nEmail : customercarejammu@gmail.com\nP

In [62]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
print(context)

(Reference to -  The diagnostic and predictive role of NLR, d-NLR and PLR in COVID-19 patients
Recommendations
(Reference to -  The diagnostic and predictive role of NLR, d-NLR and PLR in COVID-19 patients   A.-P. Yang, et al.  International Immunopharmacology 84 (2020) 106504
This ratio element is a calculated parameter and out of NABL scope.
View Details View Report
Page 2 Of 11
Dr. Shavi Mahajan
Labhead-Pathologist
ULR No.775000013478100-0121
Agilus Diagnostics Ltd
Astro Healthcare R&D Pvt. Ltd., Lab 28-C, 2nd Extension, Opp. Power Grid Corporation,
Gandhi Nagar,Jammu Tawi
Jammu, 180004
Jammu & Kashmir, India
Tel : 0191-2479435,0191-2479439
918899054002, Fax : 
CIN - U74899PB1995PLC045956
Email : customercarejammu@gmail.com
PERFORMED AT :
(Reference to -  The diagnostic and predictive role of NLR, d-NLR and PLR in COVID-19 patients   A.-P. Yang, et al.  International Immunopharmacology 84 (2020) 106504
This ratio element is a calculated parameter and out of NABL scope.
View Details 

In [63]:
llm = ChatOpenAI(model="gpt-5")

In [64]:
res = llm.invoke(f"""Can you provide me the answer based on provided context for my question, 
                context:{context} and question : {query}""")

In [65]:
print(res.content)

It looks like your context references the diagnostic/predictive value of NLR, d‑NLR, and PLR in COVID‑19 and notes that these ratios are calculated (and “out of NABL scope”). If you’re looking for Machine Learning and Data Science content grounded in that context, here’s a concise, practical blueprint you can use to build and evaluate models that incorporate these ratios.

What the ratios are (as ML features)
- NLR (Neutrophil-to-Lymphocyte Ratio) = absolute neutrophils / absolute lymphocytes
- d-NLR (derived NLR) = absolute neutrophils / (total WBC − absolute lymphocytes)
- PLR (Platelet-to-Lymphocyte Ratio) = platelets / absolute lymphocytes
- Rationale: These are inflammation/immune response proxies that have shown association with COVID-19 severity and outcomes in multiple studies, including the referenced paper.

Typical ML use cases
- Early risk stratification: predict severe disease, ICU admission, need for ventilation, or in‑hospital mortality at admission or within a defined h

#### CHAIN - Context_generate | prompt | llm | strparser

In [73]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data : 
        context += doc.page_content + "\n"
        
    return {
        "context":context,
        "question":query
    }

In [74]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answers based on the context for our question. In case 
    you do not know the answer , then you can say that 'I do not know.'
    Context:{context}
    Question:{question}
""")

In [75]:
rag_chain = get_context | prompt | llm

In [76]:
res = rag_chain.invoke("What are the body tests that I have gone through")
print(res.content)

Based on the report, you had the following biochemistry tests:

- Fasting Blood Sugar (Glucose, fluoride plasma)
- Kidney Function Tests (serum), including:
  - Blood Urea Nitrogen (BUN)
  - Creatinine
  - BUN/Creatinine Ratio
  - Uric Acid
  - Total Protein
  - Albumin
  - Globulin


In [70]:
res = rag_chain.invoke("What is the duration of Machine learning Course?")
print(res.content)

I do not know. The provided context does not mention a Machine Learning course or its duration.


In [78]:
res = rag_chain.invoke("Give me the result of Kidney Test.")
print(res.content)

Here are your kidney-related test results from the provided report:

- Serum creatinine: 0.69 mg/dL (reference 0.90–1.30) — flagged low
- BUN/Creatinine ratio: 14.49 (reference 10–20)
- Uric acid: 4.9 mg/dL (reference 3.7–9.2)
- Electrolytes: Sodium 139 mmol/L (136–145), Potassium 4.12 mmol/L (3.5–5.1), Chloride 107 mmol/L (98–107)
- Calcium: 9.2 mg/dL (8.3–10.6)
- Urinalysis:
  - Nitrite: Negative
  - RBC: Not detected
  - WBC (pus cells): Not detected
  - Epithelial cells: 1/HPF
  - Casts: Not detected
  - Crystals: Not detected
  - Bacteria: Not detected

Summary: Kidney-related parameters and urinalysis are within normal limits overall. The creatinine is slightly below the lab’s reference range, which is commonly seen with lower muscle mass and is not typically a sign of kidney disease. If you want an estimated GFR (a key kidney function measure), it wasn’t provided; I can help estimate it if you share your age and sex.
